<center><b>CTS2000: Scripting for the Humanities</b></center>
<center>7-2: Historical Named Entity Recognition with Stanza</center>

---

# Named Entity Recognition with Stanza

## Overview

First we started with one letter and then we scaled our text analysis tools to the whole Montagu corpus, and along the way we ran into POS tagging and lemmatization. This notebook introduces a related but different task: **Named Entity Recognition (NER)**.

Where POS tagging asks "what part of speech is this word?", NER asks "does this word refer to a person, a place, an organization, a date, or something else in the world?"

We will follow the same pattern as before: start with one letter, understand the tool deeply, then scale up to the full corpus. New to this notebook:

1. **Setup**: installing and loading Stanza
2. **NER on a single sentence**: understanding entity types and the output format
3. **NER on Letter 35**: applying it to a full letter
4. **A key limitation**: why a model trained on modern news text struggles with 18th-century letters
5. **Scaling to the corpus**: running NER on all letters and storing results
6. **Aggregating entities**: counting people and places across the corpus

---

**Dataset:** Lady Mary Wortley Montagu's *Turkish Embassy Letters* (1716–1718)

---


## Step 1: Setup

[Stanza](https://stanfordnlp.github.io/stanza/ner.html) is a different library from NLTK, so it has its own installation and download steps. Unlike NLTK, where we downloaded small individual resources (`punkt`, `stopwords`, etc.), Stanza downloads a full **pipeline** which is a bundle of neural network models that work together (tokenizer, POS tagger, NER tagger, and so on).

[PyTorch](https://pypi.org/project/torch/) is a machine learning framework for Python used primarily for deep learning applications like computer vision and natural language processing. It is one of the [most important deep learning frameworks](https://pytorch.org/) today.



In [ ]:
import torch
# Stanza's official models need weights_only=False on torch 2.2 
_torch_load = torch.load
torch.load = lambda *args, **kwargs: _torch_load(*args, **{**kwargs, "weights_only": False})

import stanza
import pandas as pd
import re
from collections import Counter
from itertools import chain


In [ ]:
stanza.download('en') # download English model

In [ ]:
# Let's test the example from the repository itself
# https://github.com/stanfordnlp/stanza
# This allows us to test that the code is working as intended.

nlp = stanza.Pipeline('en') # This sets up a default neural pipeline in English
doc = nlp("Barack Obama was born in Hawaii. He was elected president in 2008.")
doc.sentences[0].print_dependencies()

> **What just happened?** `stanza.Pipeline(...)` loaded the models into memory and built a function-like object, `nlp`, that we can now call on any piece of text. This loading step is the slow part — once `nlp` exists, processing text with it is much faster. This is why we build the pipeline once, outside of any loop, rather than rebuilding it for every letter.

In [ ]:
print(doc)

1. There are two sentences in this example, even though we gave Stanza a single string. First, Stanza **split the text into sentences**:
    - Sentence 1: Barack Obama was born in Hawaii.
    - Sentence 2: He was elected president in 2008.

2. Each sentence was then **tokenized**: split into words and punctuation. Each token gets an `id` (its position in the sentence), its `text`, and `start_char` and `end_char`, which record where it sits in the original string.

    Next comes the **mwt** (multi-word token) step. Some tokens are really two words fused together, like *wasn't* (*was* + *not*) or *I'm* (*I* + *am*). The mwt step splits these apart. Our example sentences have none, so this step didn't change anything.

    Here is the first token of the first sentence, with everything Stanza recorded about it:

    ```json
    "id": 1,
    "text": "Barack",
    "lemma": "Barack",
    "upos": "PROPN",
    "xpos": "NNP",
    "feats": "Number=Sing",
    "head": 4,
    "deprel": "nsubj:pass",
    "start_char": 0,
    "end_char": 6,
    "ner": "B-PERSON",
    "multi_ner": [
        "B-PERSON"
    ]
    ```

    The rest of the steps each add a few of these fields.

3. The tokens were **pos**-tagged. Stanza gives two kinds of part-of-speech tag, and a list of grammatical features:

    ```json
    "upos": "PROPN",
    "xpos": "NNP",
    "feats": "Number=Sing",
    ```

    - `upos` is the **universal** POS tag. It uses the same small set of tags for every language. `PROPN` means proper noun.
    - `xpos` is the **language-specific** tag. For English, these are the same Penn Treebank tags we saw with NLTK's `pos_tag`. `NNP` means singular proper noun.
    - `feats` lists grammatical **features**, such as number, tense or person. *Barack* is singular. Compare *was* (id:3), which has five: `Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin`.

4. The tokens were **lemma**tized.

    ```json
    "lemma": "Barack",
    ```

    A name is already in its base form, so the lemma is the same as the text. The verbs are more interesting: the lemma of *born* (id:4) is *bear*, and the lemma of *was* (id:3) is *be*.

5. The sentences were **dependency parsed** (**depparse**). This step works out how the words relate to each other grammatically.

    ```json
    "head": 4,
    "deprel": "nsubj:pass",
    ```

    - `head` is the `id` of the word this one depends on. Token 4 is *born*.
    - `deprel` is the relation between them. `nsubj:pass` means *Barack* is the passive subject of *born*: the person the being-born happened to.

    Every word has exactly one head, except the main word of the sentence, the **root**. *Born* (id:4) is the root of sentence 1, so its head is `0`.

6. Finally, Stanza ran **named entity recognition** (**ner**).

    ```json
    "ner": "B-PERSON",
    "multi_ner": [
        "B-PERSON"
    ]
    ```

    The tag has two parts. `PERSON` is the entity type. The letter before it says where this token sits in the entity:

    - `B`: the **beginning** of a multi-word entity
    - `I`: **inside** one
    - `E`: the **end** of one
    - `S`: a **single**-token entity
    - `O`: **outside** any entity

    So *Barack* is `B-PERSON` and *Obama* is `E-PERSON`: together, one person. *Hawaii* is `S-GPE`, a one-word geopolitical entity.

    `multi_ner` matters only if you run several NER models at once. With one model, it just repeats `ner`.

Let's take a closer look to these sentences:

![img](../../img/stanza_sentence_1_all_layers.png)

![img](../../img/stanza_sentence_2.png)

In [ ]:
# processors='tokenize,ner' loads only what we need:
# - tokenize: splits text into words and sentences (NER depends on this)
# - ner: the named entity recognizer itself

# We could also run this instead of all the stanza models to only load the tokenizer, which we need to be able to run the NER and the NER model

ner = stanza.Pipeline(lang='en', processors='tokenize,ner')

---
## Step 2: NER on a single sentence

Before we turn to a the letters, let's pick a sentence from one letter and run NER on it

In [ ]:
# it is a sentence from letter 32
sample = "My side-saddle is the first that was ever seen in this part of the world, and is gazed at with as much wonder as the ship of Columbus in the first discovery of America."

doc_sample = nlp(sample)

for sentence in doc_sample.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}") #!r:25 makes print statement look neat like a table

Let's unpack this:

`doc_sample = nlp(sample)` -> running our pipeline on a string returns a `Document` object, not a plain string or list. This is different from NLTK, where most functions returned lists or tuples directly.

`doc_sample.sentences` -> a `Document` is broken into `Sentence` objects, even if your input is one sentence. This matters for the corpus-level work later, since Stanza's own sentence splitting can disagree with NLTK's `sent_tokenize()`.

`sentence.ents` -> each `Sentence` has an `.ents` attribute: a list of entities Stanza found in it.

`ent.text` and `ent.type` -> each entity has the original text span (`ent.text`) and a category label (`ent.type`), such as `PERSON`, `GPE` (geo-political entity — countries, cities, states), `ORG`, or `DATE`.

In [ ]:
# this print statement from stanza documentation shows us each word in the sentence and how it was/was not tagged

print(*[f'token: {token.text}\tner: {token.ner}' for sent in doc_sample.sentences for token in sent.tokens], sep='\n')

### Entity types you'll commonly see

| Type | Meaning | Example |
|---|---|---|
| `PERSON` | People, including fictional | *Obama, Montagu* |
| `GPE` | Countries, cities, states | *Hawaii, Constantinople* |
| `LOC` | Non-GPE locations: mountains, bodies of water, regions | *the Bosphorus* |
| `ORG` | Companies, agencies, institutions | *the Senate* |
| `DATE` | Absolute or relative dates | *1717, last Tuesday* |
| `NORP` | Nationalities, religious or political groups | *French, Catholic* |
| `FAC` | Buildings, airports, bridges | *the Globe Theatre* |


> Stanza's English NER model is trained on the **[OntoNotes](https://catalog.ldc.upenn.edu/LDC2013T19)** dataset, a large collection of modern news, talk shows, web text, and similar sources. They have 18 categories for entities, which you can read more about [here on page 21](https://catalog.ldc.upenn.edu/docs/LDC2013T19/OntoNotes-Release-5.0.pdf)

---
## Step 3: NER on Letter 35

Let's bring back Letter 35, the same letter we worked with closely in 2-4 and again at the start of 3-1.

In [ ]:
# Load the corpus
montagu = pd.read_csv('../../data/montagu/montagu_letters_v3.csv')
montagu = montagu[['filename', 'title', 'body']]
montagu.head()


In [ ]:
# eding the \n line breaks from the body for stanza
montagu['body'] = montagu['body'].apply(lambda text: " ".join(text.split()))

In [ ]:
letter_35 = montagu['body'].iloc[34]
print(letter_35[:300])

In [ ]:
# Run the Stanza pipeline on the full letter
doc_35 = nlp(letter_35)

In [ ]:
print(f"Number of sentences Stanza found: {len(doc_35.sentences)}")

> Compare this sentence count to the one we got from NLTK's `sent_tokenize()` (68 sentences for Letter 35). Are they the same? If not, why might two different tools disagree about where sentences begin and end?

In [ ]:
# Extract all entities from doc_35 and print them as (text, type)

letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]

for text, etype in letter_35_entities:
    print(f"{text!r:30} {etype}")

Let's unpack the list comprehension:

```python
letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]
```

This is a **nested** list comprehension — two `for` clauses instead of one. Read it left to right, but think about it inside-out:

`for sentence in doc_35.sentences` -> outer loop, over each sentence in the document

`for ent in sentence.ents` -> inner loop, over each entity within that sentence

`(ent.text, ent.type)` -> what we keep from each entity: the text span and its label

This is equivalent to:

```python
letter_35_entities = []
for sentence in doc_35.sentences:
    for ent in sentence.ents:
        letter_35_entities.append((ent.text, ent.type))
```

Both versions do the same thing. The nested comprehension is more compact once you're comfortable with it, but there's no shame in writing the longer version first and condensing it later.

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Look closely at the output above. Are there any entities that are clearly wrong — mislabeled, or not entities at all? Are there any real people or places in the letter that Stanza missed entirely?
</div>

---
## Step 4: A key limitation — why this matters for historical text

Stanza's English NER model was trained on OntoNotes: modern news wire, broadcast conversation, web text, and similar genres from the late 20th and early 21st century. Montagu was writing private letters in 1716–1718, describing a part of the world with person and place names with several competing English spellings that does not appear in the training data.

Researchers who work on **historical NER** as a subfield have documented this gap extensively. The challenges they describe consistently include archaic spelling and inconsistent orthography, named entities that no longer exist or have changed meaning, and a basic shortage of historical text to train on compared to the enormous amount of modern text available online. None of this is a flaw in Stanza specifically — it's a structural feature of how all modern NER models get trained, and it generalizes to spaCy, FLAIR, or any other off-the-shelf system you might reach for.

> This is not a reason to abandon NER for historical text. It's a reason to treat its output as a **starting point for inquiry**, not a finished answer. We need to read the output critically to ensure that we are actually we are actually capturing what is valuable and interesting to us.

In [ ]:
# Let's look at a sentence built specifically to surface this gap.
# Try swapping in a sentence or two of your own choosing from Letter 35.

test_sentence = "We lay that night at a town called Bujuk Cekmege, or Great Bridge; and the night following, at Kujuk Cekmege, or Little Bridge; in a very pleasant lodging, formerly a monastery of dervises; having before it a large court, encompassed with marble cloisters, with a good fountain in the middle."

doc_test = nlp(test_sentence)
for sentence in doc_test.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

---
## Step 5: Scaling to the corpus

Now that we have looked closely at one letter, let's apply the same pipeline to all of Montagu's letters.

> ⚠️ Running a neural pipeline on every letter in the corpus is much slower than the simple string operations. Expect this cell to take noticeably longer to run. This is normal — Stanza is doing far more computation per word than `.split()` or even NLTK's `word_tokenize()`.

In [ ]:
# Run NER on every letter and store the resulting Document object
montagu['ner_doc'] = montagu['body'].apply(lambda text: nlp(text))

It took ca. 24 minutes to run!

We've stored the full `Document` object for each letter in ner_doc, not just the entities. This mirrors what we did with `pos_tags`: keep the intermediate result around so you don't have to recompute it if you want to extract something different from it later.

Now let's pull just the entities out into their own column.

In [ ]:
def extract_entities(doc):
    """Extract all (text, type) entity pairs from a Stanza Document."""
    return [(ent.text, ent.type) for sentence in doc.sentences for ent in sentence.ents]

montagu['entities'] = montagu['ner_doc'].apply(extract_entities)
montagu[['title', 'entities']].head()

In [ ]:
montagu.info()

> Side note: We actually ran the full nlp pipeline on all letters and not the ner only pipeline. (I forgot I switched the variable names). This is not a bad thing, we just did more work and we might as well save it better for the future.

In [ ]:
# Let's extract our stanza annotations
montagu['sentences']   = montagu['ner_doc'].apply(lambda doc: [s.text for s in doc.sentences])
montagu['num_sents']   = montagu['sentences'].apply(len)
montagu['words']       = montagu['ner_doc'].apply(lambda doc: [w.text for s in doc.sentences for w in s.words])
montagu['lemmas']      = montagu['ner_doc'].apply(lambda doc: [w.lemma for s in doc.sentences for w in s.words])
montagu['pos_tags']    = montagu['ner_doc'].apply(lambda doc: [(w.text, w.upos) for s in doc.sentences for w in s.words])
montagu['xpos_tags']   = montagu['ner_doc'].apply(lambda doc: [(w.text, w.xpos) for s in doc.sentences for w in s.words])  # Penn Treebank tags, like NLTK's
montagu['morph_feats'] = montagu['ner_doc'].apply(lambda doc: [(w.text, w.feats) for s in doc.sentences for w in s.words])  # tense, number, etc.

# Dependencies: (word, relation, head word)
montagu['dependencies'] = montagu['ner_doc'].apply(
    lambda doc: [(w.text, w.deprel, s.words[w.head - 1].text if w.head > 0 else 'ROOT')
                 for s in doc.sentences for w in s.words])

# Sentiment per sentence: 0 = negative, 1 = neutral, 2 = positive
montagu['sentiment']      = montagu['ner_doc'].apply(lambda doc: [s.sentiment for s in doc.sentences])
montagu['mean_sentiment'] = montagu['sentiment'].apply(lambda x: sum(x) / len(x))

# Constituency parse trees, saved as strings
montagu['constituency'] = montagu['ner_doc'].apply(lambda doc: [str(s.constituency) for s in doc.sentences])

In [ ]:
montagu.head()

In [ ]:
# doc.to_dict() turns each Document into plain lists/dicts (tokens, lemmas, POS, deps, NER tags, etc)
# sentiment and constituency trees are not included, so we grab those separately
montagu['doc_dict'] = montagu['ner_doc'].apply(lambda doc: doc.to_dict())

# pickle keeps Python objects intact (CSV would turn lists into strings)
montagu.drop(columns=['ner_doc']).to_pickle('../../data/montagu/montagu_stanza_full.pkl')


In [ ]:
# We could read this in later with the Stanza document

# from stanza.models.common.doc import Document

# montagu = pd.read_pickle('../../data/montagu/montagu_stanza_full.pkl')
# montagu['ner_doc'] = montagu.apply(lambda row: Document(row['doc_dict'], text=row['body']), axis=1)

In [ ]:
# Save a smaller CSV to explore later. We leave out the Stanza objects and the huge nested columns.

montagu_to_save = montagu.drop(columns=['ner_doc', 'doc_dict', 'constituency'])
montagu_to_save.to_csv('../../data/montagu/montagu_letters_v4.csv', index=False)
print("Saved! Columns now:", montagu_to_save.columns.tolist())


In [ ]:
# and here is how we would access our csv later

montagu = pd.read_csv("../../data/montagu/montagu_letters_v4.csv")

montagu.head()

---
## Step 6: Aggregating entities across the corpus

With entities extracted for every letter, we can ask corpus-level questions: who does Montagu mention most? What places recur across her letters?

### People

In [ ]:
montagu["entities"].head()

In [ ]:
# Flatten all entities into one list, same pattern as chain.from_iterable in 3-1
all_entities = list(chain.from_iterable(montagu['entities']))

# Keep only PERSON entities
people = [text for text, etype in all_entities if etype == 'PERSON']
person_counts = Counter(people)

print("Top 20 most-mentioned people:")
for name, count in person_counts.most_common(20):
    print(f"  {name}: {count}")

**Why did we get this error?**

We saved our data as a csv and that coverted everything to a string. What looks like to us as a list of tuples is actually just one whole string that we converted to a list in `all_entities = list(chain.from_iterable(montagu['entities']))`.

This does not mean however that everything else got turned into the correct data types.

Don't worry, there is a fix!!

In [ ]:
type(all_entities)

In [ ]:
type(all_entities[0])

In [ ]:
print(all_entities[0])

In [ ]:
print(all_entities[:20])

`ast` is a Python module, Abstract Syntax Trees. `ast.literal_eval` takes a string of Python and turns it inot the real value it describes

In [ ]:
import ast

all_entities = list(chain.from_iterable(ast.literal_eval(ents) for ents in montagu['entities']))


In [ ]:
type(all_entities)

In [ ]:
type(all_entities[0])

In [ ]:
print(all_entities[0])

In [ ]:
# Keep only PERSON entities
people = [text for text, etype in all_entities if etype == 'PERSON']
person_counts = Counter(people)

print("Top 20 most-mentioned people:")
for name, count in person_counts.most_common(20):
    print(f"  {name}: {count}")

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Scan this list. Are these all genuinely distinct people, or are they even all people?
</div>

### Places

What counts as a place for Stanza?

- GPE (geo-political entity) covers cities and countries.
- LOC covers rivers, seas and regions.

Let's extract GPE and LOC entities and count the most common ones.

In [ ]:
places = [text for text, etype in all_entities if etype in ('GPE', 'LOC')]
place_counts = Counter(places)

print("Top 20 most-mentioned places:")
for place, count in place_counts.most_common(20):
    print(f"  {place}: {count}")

In [ ]:
# Pick one place from your list above and confirm it actually appears where you'd expect
search_term = "Vienna"  # change this to check on a new term

matches = montagu[montagu['body'].str.contains(search_term, case=False, na=False)]
print(f"'{search_term}' appears in {len(matches)} letter(s):")
print(matches['title'].tolist())

People will have to wait for the next notebook but places seem good to go for a first try!

## Can we map these places?

Yes! but it requires a few steps.

First, we need to decide what we are exactly visualizing. There are a few options.

*We have:* places mentioned in each letter.

*We can visualize:* in how many letters a given place is mentioned.

Let's get this into a more useable format.

*Step 1:* Create a new column in our df that for each letter shows the placename and its counts.

In [ ]:
# remember our string problem?

montagu['entities'] = montagu['entities'].apply(ast.literal_eval)

In [ ]:
montagu["places"] = montagu["entities"].apply(
    lambda ents: Counter(text for text, etype in ents if etype in ('GPE', 'LOC'))
)

montagu[["title", "places"]].head()

*Step 2:* Let's get a list of all the unique places

In [ ]:
all_unique_places = sorted(set(text for text, etype in all_entities if etype in ('GPE', 'LOC')))
print(len(all_unique_places), "unique places")

205 unique places are a lot of places to map at once.

*Step 3:* Let's get a sense of how often these places appear

In [ ]:
# this is our counter for corpus wide place names
letter_counts = Counter(place for c in montagu["places"] for place in c.keys())

# mentioned only once
once = [p for p, n in place_counts.items() if n == 1]
# mentioned only twice
twice = [p for p, n in place_counts.items() if n == 2]
# mentioned twice in the same letter
twice_same_letter = [p for p in twice if letter_counts[p] == 1]
# mentioned in two different letters
twice_diff_letters = [p for p in twice if letter_counts[p] == 2]

print(f"Unique places in total: {len(place_counts)}")
print(f"Mentioned only once: {len(once)}")
print(f"Mentioned exactly twice: {len(twice)}")
print(f"  both in the same letter: {len(twice_same_letter)}  {sorted(twice_same_letter)}")
print(f"  in two different letters: {len(twice_diff_letters)}  {sorted(twice_diff_letters)}")

Unique places in total: 205

Mentioned only once: 142

Mentioned exactly twice: 25

  both in the same letter: 9  ['Abydos', 'Candia', 'Chersonesus', 'Chios', 'Fauxburg', 'Japan', 'Poland', 'Sigaeum', 'Versailles']
  
  in two different letters: 16  ['Adrianople', 'Aleppo', 'Archipelago', 'Betsko', 'Frankfort', 'Galata', 'Grand Cairo', 'Jupiter', 'Leipzig', 'Levant', 'Mecca', 'Morea', 'Prague', 'Sophia', 'Venice', 'china']

Interesting! Two take aways:

- 142 out of 205 places are only mentioned once.
- Perhaps there are some spelling errors? It is hard to imagine Adrianople was mentioned only in two letters.

*Step 4:* Let's investigate Adrianople

In [ ]:
place = "Adrianople"

# 1. Ground truth: how often does the word appear in the raw letters?
raw_hits = montagu["body"].str.count(place)
print(f"In the raw text: {raw_hits.sum()} mentions in {(raw_hits > 0).sum()} letters")

# 2. What did Stanza call it?
labels = Counter(etype for ents in montagu["entities"] for text, etype in ents if text == place)
print("Stanza's labels:", labels)

# 3. Look for spelling variants, just in case
variants = Counter(w for body in montagu["body"] for w in re.findall(r"\b[Aa]dri\w*", body))
print("Variants in the text:", variants)


Aha! The issue was Stanza's categorization, which is to be expected with an NER model trained on modern texts.

Before we fix this, we need to run the same check on other places we expect to be common, like Constantinople, Pera or Belgrade, to see whether the mislabelling happens elsewhere. "Constantinople" and "Sancta Sophia" already showed up as PERSON top 20.

*Step 5:* Let's fix some placenames semi-automatically

In [ ]:
# I did a first pass and chose these

suspect_places = ["Constantinople", "Adrianople", "Peterwaradin", "Cairo", "St Pierre d'Arena"]

for place in suspect_places:
    raw = montagu["body"].str.count(re.escape(place)).sum()
    labels = Counter(etype for ents in montagu["entities"] for text, etype in ents if text == place)
    as_place = labels["GPE"] + labels["LOC"]
    print(f"{place:20} raw text: {raw:3}   tagged: {sum(labels.values()):3}   "
          f"as GPE/LOC: {as_place:3}   labels: {dict(labels)}")


In [ ]:
# while working on that, I also realized that a few places were not split correctly

merge_map = {"GENOA": "Genoa", "Old England": "England", "Grand Cairo": "Cairo"}

for variant, standard in merge_map.items():
    for name in (variant, standard):
        raw = montagu["body"].str.count(re.escape(name)).sum()
        labels = Counter(etype for ents in montagu["entities"] for text, etype in ents if text == name)
        as_place = labels["GPE"] + labels["LOC"]
        arrow = "  ->" if name == standard else ""
        print(f"{arrow:4}{name:16} raw text: {raw:3}   tagged: {sum(labels.values()):3}   "
              f"as GPE/LOC: {as_place:3}   labels: {dict(labels)}")
    print()

In [ ]:
montagu["places_fixed"] = montagu["entities"].apply(
    lambda ents: Counter(merge_map.get(text, text) for text, etype in ents
                         if etype in ("GPE", "LOC") or text in suspect_places)
)

# Let's see what we changed
changed = montagu[montagu["places"] != montagu["places_fixed"]]
changed[["title", "places_fixed"]]

In [ ]:
# this is our UPDATED counter for corpus wide place mentions
place_counts_updated = sum(montagu["places_fixed"], Counter())

# this is our UPDATED counter for how many letters each place appears in
letter_counts_updated = Counter(place for c in montagu["places_fixed"] for place in c.keys())

# mentioned only once
once = [p for p, n in place_counts_updated.items() if n == 1]
# mentioned only twice
twice = [p for p, n in place_counts_updated.items() if n == 2]
# mentioned twice in the same letter
twice_same_letter = [p for p in twice if letter_counts_updated[p] == 1]
# mentioned in two different letters
twice_diff_letters = [p for p in twice if letter_counts_updated[p] == 2]

print(f"Unique places in total: {len(place_counts_updated)}")
print(f"Mentioned only once: {len(once)}")
print(f"Mentioned exactly twice: {len(twice)}")
print(f"  both in the same letter: {len(twice_same_letter)}  {sorted(twice_same_letter)}")
print(f"  in two different letters: {len(twice_diff_letters)}  {sorted(twice_diff_letters)}")


We could investigate this further, like `china` in lowercase looks suspicious to me because it is more likely to be referencing an object made of porcelain. We will take a closer look at this in another notebook. 

*Step 6:* Let's get a list of unique places after keeping places that appear in at least two letters and mentioned at least 3 times

In [ ]:
recurring_places = sorted(
    (p for p in place_counts_updated
     if letter_counts_updated[p] >= 2 and place_counts_updated[p] >= 3),
    key=lambda p: -place_counts_updated[p]
)

print(f"{len(recurring_places)} places")

In [ ]:
for p in recurring_places:
    print(f"  {p:20} {place_counts_updated[p]:3} mentions in {letter_counts_updated[p]:2} letters")

35 places! Very manageable and interesting!

*Step 7:* Read the letters and evaluate these places

In [ ]:
def kwic(term, width=50):
    """Print every mention of term, with `width` characters of context on each side."""
    for title, body in zip(montagu["title"], montagu["body"]):
        for match in re.finditer(re.escape(term), body):
            left = body[max(0, match.start() - width):match.start()].replace("\n", " ")
            right = body[match.end():match.end() + width].replace("\n", " ")
            print(f"{title:10} {left:>{width}} [{match.group()}] {right}")

In [ ]:
kwic("St Lawrence", width=100)

Let's unpack this:

`def kwic(term, width=50):` -> a function with two inputs: the word we're looking for, and how many characters of context to show on each side. `width=50` is a **default value**: if we don't give a width, Python uses 50. So `kwic("St Lawrence")` and `kwic("St Lawrence", width=50)` do the same thing, but we can also write `kwic("St Lawrence", width=100)` to see more context.

`for title, body in zip(montagu["title"], montagu["body"]):` -> `zip` pairs up the two columns, so on each loop we get one letter's title and its text together. This lets us print which letter each mention comes from.

`for match in re.finditer(re.escape(term), body):` -> `re.finditer` finds **every** match in the letter, not just the first one. Each `match` remembers where it starts and ends in the text.

`re.escape(term)` -> in regular expressions, some characters have special meanings. For example, `.` means "any character". `re.escape` tells Python to search for our term exactly as we typed it.

`match.start()` and `match.end()` -> the position of the first character of the match, and the position just after its last character. These are like the `start_char` and `end_char` that Stanza gave us for each token.

`left = body[max(0, match.start() - width):match.start()]` -> we slice out the `width` characters just before the match. If the match is near the beginning of a letter, `match.start() - width` could be a negative number, and a negative number in a slice counts from the **end** of the text. `max(0, ...)` picks whichever is larger, so the slice never starts before 0.

`right = body[match.end():match.end() + width]` -> the same for the text just after the match. We don't need `max` here, because Python simply stops at the end of the text if the slice goes past it.

`.replace("\n", " ")` -> we replace line breaks with spaces, so each mention prints on a single line.

`print(f"{title:10} {left:>{width}} [{match.group()}] {right}")` -> this is what makes it look like a KWIC:
- `{title:10}` gives the title 10 characters of space, so the columns line up.
- `{left:>{width}}` **right-aligns** the left context. This pushes every keyword into the same column, so we can read down the middle.
- `[{match.group()}]` prints the match itself in square brackets, so it stands out.

PS: KWIC stands for **KeyWord In Context**. You can use this code or a version of it for visualizing any term that you want to show in context.

In [ ]:
# St Lawrence passed our filters, but reading the letters shows three different things:
# a convent in Vienna, a church in Genoa, and a fair in Paris. Not the same place mentioned three times no a place on her route.
drop = {"St Lawrence"}
recurring_places = [p for p in recurring_places if p not in drop]

print(f"{len(recurring_places)} places")

### Interlude

Remember how we identified the locations that the letters were sent from during our text extraction step? They will now come very handy!

In [ ]:
locations = pd.read_csv("../../data/montagu/locations.csv")
locations.head()

In [ ]:
# Montagu's spelling -> the modern name used in locations.csv
# This is from my original matching on Google Sheets. 
historical_to_modern = {
    "Constantinople": "Istanbul",
    "Adrianople": "Edirne",
    "Peterwaradin": "Petrovaradin",
    "Blankenburg": "Blankenburg am Harz",
}

In [ ]:
print("Places mentioned in the letters that are also sending locations:\n")
for place in recurring_places:
    modern = historical_to_modern.get(place, place)
    match = locations[locations["location_name"] == modern]
    if not match.empty:
        row = match.iloc[0]
        print(f"  {place:16} -> {row['location_name']:20} {row['location_id']}  {row['location_wikidata']}  "
              f"mentioned {place_counts_updated[place]} times")

In [ ]:
# let's keep track of the ones we don't have QIDs for

not_in_locations = [p for p in recurring_places
                    if historical_to_modern.get(p, p) not in set(locations["location_name"])]
not_in_locations

What do you think of these places? Are they all the same kind of place? Are they all cities or countries?

**Overview**

| Type | Places |
|---|---|
| Countries and regions | England, Hungary, Germany, France, Italy, Turkey, Bohemia, Holland, Austria, *Servia* |
| Continents | Europe, Asia |
| Rivers and mountains | Danube, Hebrus, Alps |
| Cities and towns | London, Dresden, Rome, *Philippopolis*, *Buda*, *Essek*, Cairo |


Wait, what is *Servia*?

Servia is an old spelling for Serbia. Here is an example from the Broadview edition of the letters, which I used for cross-referencing while preparing these placenames:

![img](../../img/broadview_99_servia.png)

![img](../../img/broadview_99_footnote.png)

Before we can map these places, we need their coordinates. To find coordinates, we need Wikidata QIDs (or another reliable entity linking system). To search for Wikidata QIDs, we need modern placenames. Luckily, for many modern placenames are quite similar but not for all.

Find the modern names for these places:

- Philippopolis
- Buda
- Essek

In [ ]:
historical_to_modern = {
    # sending locations (match locations.csv)
    "Constantinople": "Istanbul",
    "Adrianople": "Edirne",
    "Peterwaradin": "Petrovaradin",
    "Blankenburg": "Blankenburg am Harz",
    # other places mentioned in the letters
    "Servia": "Serbia",
    "Philippopolis": "Plovdiv",
    "Essek": "Osijek",
    "Hebrus": "Maritsa River",
    "Buda": "Budapest",
    "Holland": "Netherlands",
    "Turkey": "Ottoman Empire",
}

In [ ]:
# a small table: Montagu's name next to the modern one
place_names = pd.DataFrame({"place": recurring_places})
place_names["modern_name"] = place_names["place"].map(lambda p: historical_to_modern.get(p, p))
place_names

In [ ]:
# QIDs we already have from locations.csv
qid_lookup = dict(zip(locations["location_name"], locations["location_wikidata"]))

place_names["QID"] = place_names["modern_name"].map(qid_lookup)

place_names

In [ ]:
# this is how we can see the ones that don't have a QID from the locations.csv
place_names[place_names["QID"].isna()]

There are two ways to go about this. One, we can look up all these places **manually on Wikidata**. Two, we can use **API** calls.

Let's do this manually first, look up a few places on wikidata: 

## Wikidata API

### What is an API?

This workshop is about APIs. You may have heard this terminology in a variety of programming settings. What exactly does it mean?

* "API" stands for **Application Programming Interface**.

* Broadly defined, an API is a set of rules and procedures that facilitate interactions between computers and their applications.

* A very common type of API is the Web API, which, among other things, allows users to query a remote database over the internet.

* For example, a web service such as Reddit has many databases that may be of use to us: Posts, Users, Subreddits, etc. If we want to access some portion of these databases, it'd be helpful to have a set of rules and protocols in place to outline how we access this information. This is the motivation for an API.

* Web APIs take on a variety of formats, but the vast majority adhere to a particular style known as **Representational State Transfer** or **REST**.

* What makes these "RESTful" APIs so convenient is that we can use them to query databases using URLs.

### RESTful Web APIs Are All Around You

Consider a simple Google search:

![](../../img/google_search.png)

Ever wonder what all that extra stuff in the address bar was all about?  In this case, the full address is Google's way of sending a query to its databases asking requesting information related to the search term "golden state warriors". 

![](../../img/google_link.png)

In fact, it looks like Google makes its query by taking the search terms, separating each of them with a "+", and appending them to the link "https://www.google.com/#q=".  Therefore, we should be able to actually change our Google search by adding some terms to the URL and following the general format:

![](../../img/google_link_change.png)

Using RESTful APIs is essentially formatting these URLs so that you can get the response you want.

### Some Terminology

* **Uniform Resource Locator (URL)**: a string of characters that, when interpreted via the Hypertext Transfer Protocol (HTTP), points to a data resource, notably files written in Hypertext Markup Language (HTML) or a subset of a database.  This is often referred to as a "call".

* **HTTP Methods/Verbs**:

    + *GET*: requests a representation of a data resource corresponding to a particular URL.  The process of executing the GET method is often referred to as a "GET request" and is the main method used for querying RESTful databases.
    
    + *HEAD*, *POST*, *PUT*, *DELETE*: other common methods, though mostly never used for database querying.
    
As you might suspect from the example above, surfing the web is basically equivalent to sending a bunch of GET requests to different servers and asking for different files written in HTML.


### API or Web Scraping?

When deciding between using an API or web scraping, you should consider both the method's legality and efficiency. APIs provide structured, authorized access to data, often with clear documentation and rate limits to manage server load.

Web scraping, on the other hand, involves extracting data from web pages, which may violate a site's terms of service or lead to challenges in navigating complex page structures.

While scraping can be useful when no API is available, APIs are generally the preferred method for accessing web data due to their reliability and compliance with legal standards.

### [Wikidata API](https://www.wikidata.org/wiki/Wikidata:REST_API)

The Wikibase REST API is an OpenAPI-based interface that allows users to interact with, retrieve and edit Items, Properties and Statements on Wikibase instances -- including of course Wikidata. For more information about REST, see the Wikipedia entry on representational state transfer.

**Let's search for one place: Plovdiv**

In [ ]:
import requests

url = "https://www.wikidata.org/w/api.php"

params = {
    "action": "wbsearchentities",   # "search for items"
    "search": "Plovdiv",            # what we're looking for
    "language": "en",               # search English labels
    "format": "json",               # give us the answer as JSON
}

In [ ]:
# Wikidata asks every script to say who it is
headers = {"User-Agent": "CTS2000-class-notebook/1.0"}

response = requests.get(url, params=params, headers=headers)
data = response.json()

data["search"][0]

**What do we want to save from this?**

In [ ]:
first = data["search"][0]
print(first["id"], "|", first["label"], "|", first.get("description"))


**Now that we established that this works, let's scale it up**

In [ ]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "CTS2000-class-notebook/1.0"}

def get_qid(name):
    """Search Wikidata for a name. Returns (QID, description, status)."""
    params = {"action": "wbsearchentities", "search": name,
              "language": "en", "format": "json"}
    try:
        response = requests.get(WIKIDATA_API, params=params, headers=HEADERS)
        if response.status_code != 200:
            return None, None, f"error {response.status_code}"
        results = response.json()["search"]
        if not results:
            return None, None, "no results"
        return results[0]["id"], results[0].get("description"), "ok"
    except Exception as e:                      # anything unexpected: save it, keep going
        return None, None, f"error: {type(e).__name__}"

Let's unpack this:

This function does exactly what we did for Plovdiv, but for any name we give it. It also adds a few safety checks, because when we send dozens of requests, some of them will go wrong.

`def get_qid(name):` -> we define a function that takes one input, a place name, so we can reuse the same steps for every place on our list.

`params = {...}` -> the same search settings we used for Plovdiv. The only difference is that `"search"` is now `name`, whatever place we pass in, instead of `"Plovdiv"`.

`try:` and `except Exception as e:` -> `try` means "attempt everything in this block". If anything inside it fails (the internet drops, Wikidata sends back something unexpected), Python jumps to `except` instead of crashing. Without this, one bad request would stop the whole loop, and we'd lose our progress on every place after it.

`response = requests.get(WIKIDATA_API, params=params, headers=HEADERS)` -> the GET request, same as before. `WIKIDATA_API` and `HEADERS` are written in capitals because they are **constants**: values we set once and never change.

`if response.status_code != 200:` -> every response comes with a **status code**. `200` means "OK, here is your data". Anything else means something went wrong, so we stop here and return the code, for example `"error 429"`.

`results = response.json()["search"]` -> we turn the response into JSON and keep the list of search results, just like `data["search"]` for Plovdiv.

`if not results:` -> an empty list counts as `False` in Python, so this checks whether Wikidata found nothing. If so, we return `"no results"`.

`return results[0]["id"], results[0].get("description"), "ok"` -> if everything worked, we return three things from the **first** result: its QID, its description and the status `"ok"`. We use `.get("description")` instead of `["description"]` because some Wikidata entries have no description. `.get()` returns `None` when a key is missing, while `["description"]` would cause an error.

`f"error: {type(e).__name__}"` -> if something unexpected went wrong, we save the name of the error (like `ConnectionError`) as our status, so we can look at it later.

Notice that the function **always** returns three values, whether it succeeds or fails. When it fails, the QID and description are `None`. This is what lets us write `qid, description, status = get_qid(...)` in the loop below, and later filter for every place whose status isn't `"ok"`.

In [ ]:
# let's test our function

get_qid("Osijek")

In [ ]:
import time

for i, row in place_names[place_names["QID"].isna()].iterrows():
    qid, description, status = get_qid(row["modern_name"])
    place_names.loc[i, "QID"] = qid
    place_names.loc[i, "description"] = description
    place_names.loc[i, "status"] = status
    time.sleep(1) # be polite: don't hammer the server

place_names


In [ ]:
place_names[place_names["status"] != "ok"]


In [ ]:
place_names[place_names["status"].str.contains('error')]

|    | place  | modern_name   | QID | description | status    |
|----|--------|---------------|-----|-------------|-----------|
| 30 | Cairo  | Cairo         | NaN | NaN         | error 429 |
| 31 | Servia | Serbia        | NaN | NaN         | error 429 |
| 32 | Hebrus | Maritsa River | NaN | NaN         | error 429 |


Error 429 means that we sent in *too many requests* Wikidata is rate-limiting us.

Let's wait a bit and call our function only on these.

In [ ]:
for i, row in place_names[place_names["status"].str.contains('error')].iterrows():
    qid, description, status = get_qid(row["modern_name"])
    place_names.loc[i, "QID"] = qid
    place_names.loc[i, "description"] = description
    place_names.loc[i, "status"] = status
    time.sleep(1) # be polite: don't hammer the server


In [ ]:
place_names

Does anything stand out to you?

In [ ]:
place_names.loc[12, ["place", "description"]]

Europe is tagged as a *"database of biomedical research"*

**(How) Do we fix this?**

1. We can go fix this **manually** and see if there is a Wikidata entry for Europe, the continent (hint, there is: https://www.wikidata.org/wiki/Q46)

2. We can see if this was an issue resulting from our code.

In [ ]:
params = {"action": "wbsearchentities", "search": "Europe",
          "language": "en", "format": "json", "limit": 10}
results = requests.get(WIKIDATA_API, params=params, headers=HEADERS).json()["search"]

pd.DataFrame([{"QID": r["id"], "label": r["label"], "description": r.get("description")}
              for r in results])


Our code only saved the first result, which in this case resulted in a mismatch. Keeping the first result was a shorthand and most places in this list didn't have more than one result to begin with.

3. We can discard it.

Not because of the QID but maybe we should not keep *Europe* and *Asia* for that matter in our place_names list. They are continents and continents are too large to be a single point on our map.

There are merits to keeping continents on a placenames list but this is a decision that needs to be legitimized within a research project. We will talk more about this later but for now, we will discard the two continents.

In [ ]:
place_names = place_names[~place_names["place"].isin(["Europe", "Asia"])].reset_index(drop=True)
place_names

### Coordinates

We can also get coordinates in the same way using the Wikidata API. We just need to modify our function a little.

Why didn't we do this in one pass?

Because these are actually two separate requests, so there is no harm in doing them in two steps. Besides we wanted to make sure that the matches from place names are indeed matching to the places that we want them to. Hence the descriptions.

**Let's test this on Plovdiv again**

In [ ]:
params = {
    "action": "wbgetentities",   # "give me the data for these items"
    "ids": "Q459",               # Plovdiv
    "props": "claims",           # claims = the facts about the item
    "format": "json",
}

data = requests.get(WIKIDATA_API, params=params, headers=HEADERS).json()

# Follow the path down to the coordinates
coords = data["entities"]["Q459"]["claims"]["P625"][0]["mainsnak"]["datavalue"]["value"]
print(coords["latitude"], coords["longitude"])


Now, let's learn a different aspect of the Wikidata API: batches

In [ ]:
qids = place_names["QID"].dropna().tolist()

params = {"action": "wbgetentities", "ids": "|".join(qids),
          "props": "claims", "format": "json"}
entities = requests.get(WIKIDATA_API, params=params, headers=HEADERS).json()["entities"]

def get_coords(qid):
    """Pull latitude and longitude out of one entity. Returns (None, None) if it has none."""
    claims = entities.get(qid, {}).get("claims", {})
    if "P625" not in claims:
        return None, None
    value = claims["P625"][0]["mainsnak"]["datavalue"]["value"]
    return value["latitude"], value["longitude"]


In [ ]:
# let's call this on our df

place_names[["latitude", "longitude"]] = place_names["QID"].apply(
    lambda q: pd.Series(get_coords(q))
)
place_names

In [ ]:
place_names[place_names["latitude"].isna()]

In [ ]:
# Let's save our coordinates and QIDs
# should drop the status col

place_names.to_csv("../../data/montagu/places_for_mapping.csv")

In [ ]:
place_names = pd.read_csv("../../data/montagu/places_for_mapping.csv")
place_names.head()

Where were we before this interlude?

Yes, mapping

### A mini exercise in mapping

*What do we need for mapping?*

**Plotly** is a Python library for making interactive charts and maps that you can hover over, zoom into and explore directly in a notebook or web browser.

To display Plotly maps inside the notebook, we need to install one more library, `nbformat`, in our environment.

1. **Open a terminal.** Terminal > New Terminal
2. **Activate our class environment:** `conda activate cts2000`
You should now see `(cts2000)` at the start of the line.
3. **Install the library:** `pip install nbformat`
Wait until you see `Successfully installed nbformat`.
4. **Restart the notebook kernel.** In VS Code, click **Restart** at the top of the notebook. Then re-run the cells from the top.

In [ ]:
import plotly.express as px

# bigger dot = more mentions in the letters
place_names["mentions"] = place_names["place"].map(place_counts_updated)

fig = px.scatter_geo(
    place_names,
    lat="latitude",
    lon="longitude",
    hover_name="place",                       # Montagu's name shows when you hover
    hover_data=["modern_name", "mentions"],
    size="mentions",
    #scope="europe",                           # zoom to Europe
)
fig.update_geos(lataxis_range=[25, 60], lonaxis_range=[-10, 40]) # based on our datapoint
fig.show()


Good enough for a first pass, but not good enough for a final presentation. Still a great first attempt at mapping from historical placenames